# V6 Step 7: RP-only RanPAC Online Evaluation

This notebook runs RanPAC only for V6 Random Projection rows. The no-RP baselines generated in Step 6 are retained on disk but skipped here because they are not cancellable transformed templates.


In [2]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd

ROOT = Path(r"F:\ECG")
STEP6_SUMMARY_PATH = ROOT / "data" / "processed" / "rp_v6" / "rp_v6_summary.csv"
Y_STREAM_PATH = ROOT / "data" / "processed" / "feature_outputs_v5" / "y_stream_feature_ablation_v5.npy"

OUTPUT_DIR = ROOT / "data" / "processed" / "ranpac_v6"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "ranpac_v6_summary.csv"
VALIDATION_REPORT_PATH = OUTPUT_DIR / "ranpac_v6_validation_report.json"
NOTEBOOK_PATH = ROOT / "notebooks" / "07_v6_ranpac.ipynb"

SEED_LABELS = ["s27", "s42", "s82"]
EXPECTED_ORIGINAL_STEP6_ROWS = 138
EXPECTED_RP_EXPERIMENT_ROWS = 135
EXPECTED_NORP_SKIPPED_ROWS = 3
EXPECTED_NUM_SAMPLES = 36103
EXPECTED_NUM_CLASSES = 1019
ACTIVATION = "ReLU"
RIDGE_LAMBDA_INITIAL = 1e-3
RIDGE_LAMBDA_FALLBACK = 1.0
UPDATE_INTERVAL = 1
DENOM_EPS = 1e-12
SCORE_EXPLOSION_LIMIT = 1e12

for folder in [OUTPUT_DIR] + [OUTPUT_DIR / f"seed{label[1:]}" for label in SEED_LABELS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Notebook path =", NOTEBOOK_PATH)
print("Input Step 6 summary =", STEP6_SUMMARY_PATH)
print("Label path =", Y_STREAM_PATH)
print("Output folder =", OUTPUT_DIR)
print("activation =", ACTIVATION)
print("ridge_lambda_initial =", RIDGE_LAMBDA_INITIAL)
print("ridge_lambda_fallback =", RIDGE_LAMBDA_FALLBACK)


Notebook path = F:\ECG\notebooks\07_v6_ranpac.ipynb
Input Step 6 summary = F:\ECG\data\processed\rp_v6\rp_v6_summary.csv
Label path = F:\ECG\data\processed\feature_outputs_v5\y_stream_feature_ablation_v5.npy
Output folder = F:\ECG\data\processed\ranpac_v6
activation = ReLU
ridge_lambda_initial = 0.001
ridge_lambda_fallback = 1.0


## 1. Read V6 Step 6 summary and select RP-only rows

The full V6 Step 6 summary has 138 rows. This cell filters to `is_rp == True`, selecting 135 RP rows and skipping the 3 no-RP baselines.


In [5]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def as_bool_series(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    })


require(STEP6_SUMMARY_PATH.exists(), f"Missing Step 6 V6 summary: {STEP6_SUMMARY_PATH}")
require(Y_STREAM_PATH.exists(), f"Missing y_stream: {Y_STREAM_PATH}")

step6_summary_full = pd.read_csv(STEP6_SUMMARY_PATH)
original_step6_rows = int(len(step6_summary_full))
is_rp_flags_full = as_bool_series(step6_summary_full["is_rp"])
is_baseline_flags_full = as_bool_series(step6_summary_full["is_baseline"])

no_rp_rows_skipped = int((~is_rp_flags_full).sum())
rp_summary = step6_summary_full[is_rp_flags_full].copy().reset_index(drop=True)
rp_only_rows_selected = int(len(rp_summary))

require(original_step6_rows == EXPECTED_ORIGINAL_STEP6_ROWS, f"Expected 138 Step 6 rows, got {original_step6_rows}")
require(rp_only_rows_selected == EXPECTED_RP_EXPERIMENT_ROWS, f"Expected 135 RP rows, got {rp_only_rows_selected}")
require(no_rp_rows_skipped == EXPECTED_NORP_SKIPPED_ROWS, f"Expected 3 no-RP rows skipped, got {no_rp_rows_skipped}")
require((~as_bool_series(rp_summary["is_baseline"])).all(), "Selected RP summary still contains baseline rows")

output_paths_exist = rp_summary["output_path"].map(lambda path: Path(str(path)).exists())
require(bool(output_paths_exist.all()), "Some selected RP output_path files are missing")

y_stream = np.load(Y_STREAM_PATH).astype(np.int32, copy=False)
unique_classes = np.unique(y_stream)
num_classes = int(len(unique_classes))
require(y_stream.shape == (EXPECTED_NUM_SAMPLES,), f"Unexpected y_stream shape: {y_stream.shape}")
require(num_classes == EXPECTED_NUM_CLASSES, f"Unexpected num_classes: {num_classes}")
require(int(unique_classes.min()) == 0 and int(unique_classes.max()) == EXPECTED_NUM_CLASSES - 1, "Labels are not 0..1018")

rp_summary = rp_summary.sort_values(["seed", "feature_alias", "rp_method_full", "projection_dim"]).reset_index(drop=True)

print("Original Step 6 summary rows =", original_step6_rows)
print("RP-only rows selected =", rp_only_rows_selected)
print("No-RP rows skipped =", no_rp_rows_skipped)
print("y_stream shape =", tuple(y_stream.shape))
print("num_classes =", num_classes)
print("Selected seeds =", sorted(rp_summary["seed_label"].unique().tolist()))


Original Step 6 summary rows = 138
RP-only rows selected = 135
No-RP rows skipped = 3
y_stream shape = (36103,)
num_classes = 1019
Selected seeds = ['s27', 's42', 's82']


## 2. Metric helper functions

These helpers compute macro precision/F1 without depending on scikit-learn. Score NaN values are expected early in the stream before any class has been seen.


In [8]:
def compute_macro_precision_f1(true_labels, predicted_labels, counted_flags, num_classes):
    true_labels = np.asarray(true_labels, dtype=np.int32)
    predicted_labels = np.asarray(predicted_labels, dtype=np.int32)
    counted_flags = np.asarray(counted_flags, dtype=bool)

    if counted_flags.sum() == 0:
        return np.nan, np.nan

    y_true = true_labels[counted_flags]
    y_pred = predicted_labels[counted_flags]
    valid_pred = (y_pred >= 0) & (y_pred < num_classes)

    true_count = np.bincount(y_true, minlength=num_classes).astype(np.float64)
    pred_count = np.bincount(y_pred[valid_pred], minlength=num_classes).astype(np.float64)
    tp_count = np.bincount(y_true[valid_pred & (y_true == y_pred)], minlength=num_classes).astype(np.float64)

    precision = np.zeros(num_classes, dtype=np.float64)
    recall = np.zeros(num_classes, dtype=np.float64)
    f1 = np.zeros(num_classes, dtype=np.float64)

    precision_mask = pred_count > 0
    recall_mask = true_count > 0
    precision[precision_mask] = tp_count[precision_mask] / pred_count[precision_mask]
    recall[recall_mask] = tp_count[recall_mask] / true_count[recall_mask]

    denom = precision + recall
    f1_mask = denom > 0
    f1[f1_mask] = 2.0 * precision[f1_mask] * recall[f1_mask] / denom[f1_mask]

    return float(np.mean(precision)), float(np.mean(f1))


def max_non_true_score_from_scores(scores, true_label):
    best_index = int(np.argmax(scores))
    best_score = float(scores[best_index])
    if best_index != int(true_label):
        return best_score

    true_value = scores[true_label].copy()
    scores[true_label] = -np.inf
    second_best = float(np.max(scores))
    scores[true_label] = true_value
    return second_best


## 3. Define one fresh RanPAC run

Every selected RP experiment gets its own fresh RanPAC state. The protocol is prequential test-then-train with ReLU activation and ridge lambda 1e-3, falling back to 1.0 only if numerical instability is detected.


In [11]:
def seed_output_dir(seed_label):
    return OUTPUT_DIR / f"seed{str(seed_label).replace('s', '')}"


def run_ranpac_once(row, y_stream, ridge_lambda, ridge_fallback_used=False, fallback_note=""):
    experiment_start = time.time()

    experiment_id = str(row["experiment_id"])
    x_rp_path = Path(str(row["output_path"]))
    projection_dim = int(row["projection_dim"])
    output_dim = int(row["output_dim"])
    seed = int(row["seed"])
    seed_label = str(row["seed_label"])
    output_folder = seed_output_dir(seed_label)
    prediction_log_path = output_folder / f"pred_{experiment_id}.csv"
    score_output_path = output_folder / f"score_{experiment_id}.npz"

    X_rp = np.load(x_rp_path)
    expected_shape = (EXPECTED_NUM_SAMPLES, output_dim)
    if X_rp.shape != expected_shape:
        raise RuntimeError(f"Unexpected X_rp shape for {experiment_id}: {X_rp.shape}, expected {expected_shape}")
    if not np.isfinite(X_rp).all():
        raise RuntimeError(f"Input X_rp contains NaN or Inf: {x_rp_path}")

    H = np.maximum(X_rp, 0).astype(np.float32, copy=False)
    del X_rp

    num_samples = int(H.shape[0])
    hidden_dim = int(H.shape[1])

    P = (np.eye(hidden_dim, dtype=np.float64) / float(ridge_lambda))
    W_out = np.zeros((hidden_dim, EXPECTED_NUM_CLASSES), dtype=np.float32)
    seen_class_counts = np.zeros(EXPECTED_NUM_CLASSES, dtype=np.int32)
    seen_classes = np.empty(EXPECTED_NUM_CLASSES, dtype=np.int32)
    num_seen_classes = 0

    predicted_labels = np.full(num_samples, -1, dtype=np.int32)
    correct_flags = np.zeros(num_samples, dtype=bool)
    seen_class_before_flags = np.zeros(num_samples, dtype=bool)
    strict_counted_flags = np.ones(num_samples, dtype=bool)
    enrollment_aware_counted_flags = np.zeros(num_samples, dtype=bool)
    true_scores = np.full(num_samples, np.nan, dtype=np.float32)
    max_non_true_scores = np.full(num_samples, np.nan, dtype=np.float32)
    running_strict_accuracy = np.zeros(num_samples, dtype=np.float32)
    running_enrollment_aware_accuracy = np.full(num_samples, np.nan, dtype=np.float32)

    strict_correct_count = 0
    enrollment_correct_count = 0
    enrollment_eval_count = 0

    for sample_index in range(num_samples):
        true_label = int(y_stream[sample_index])
        h = H[sample_index]

        scores = h @ W_out
        if not np.isfinite(scores).all() or float(np.max(np.abs(scores))) > SCORE_EXPLOSION_LIMIT:
            raise FloatingPointError(f"Unstable scores at sample {sample_index} in {experiment_id}")

        seen_class_before = seen_class_counts[true_label] > 0
        seen_class_before_flags[sample_index] = seen_class_before
        enrollment_aware_counted_flags[sample_index] = seen_class_before

        if num_seen_classes == 0:
            predicted_label = -1
            correct = False
        else:
            current_seen_classes = seen_classes[:num_seen_classes]
            predicted_label = int(current_seen_classes[int(np.argmax(scores[current_seen_classes]))])
            correct = predicted_label == true_label
            true_scores[sample_index] = scores[true_label]
            max_non_true_scores[sample_index] = max_non_true_score_from_scores(scores, true_label)

        predicted_labels[sample_index] = predicted_label
        correct_flags[sample_index] = correct

        strict_correct_count += int(correct)
        running_strict_accuracy[sample_index] = strict_correct_count / float(sample_index + 1)

        if seen_class_before:
            enrollment_eval_count += 1
            enrollment_correct_count += int(correct)
        if enrollment_eval_count > 0:
            running_enrollment_aware_accuracy[sample_index] = enrollment_correct_count / float(enrollment_eval_count)

        Ph = P @ h
        denom = 1.0 + float(h @ Ph)
        if (not math.isfinite(denom)) or denom <= DENOM_EPS:
            raise FloatingPointError(f"Unstable denominator at sample {sample_index} in {experiment_id}: {denom}")

        k = Ph / denom
        k_for_w = k.astype(np.float32, copy=False)
        error = -scores
        error[true_label] += 1.0
        W_out += k_for_w[:, None] * error[None, :]
        P -= k[:, None] * Ph[None, :]

        if sample_index % 1000 == 0 or sample_index == num_samples - 1:
            if (not np.isfinite(W_out).all()) or (not np.isfinite(P).all()):
                raise FloatingPointError(f"P or W_out became non-finite at sample {sample_index} in {experiment_id}")

        if seen_class_counts[true_label] == 0:
            seen_classes[num_seen_classes] = true_label
            num_seen_classes += 1
        seen_class_counts[true_label] += 1

    strict_prequential_accuracy = float(correct_flags.mean())
    enrollment_aware_accuracy = float(enrollment_correct_count / enrollment_eval_count) if enrollment_eval_count > 0 else np.nan
    strict_macro_precision, strict_macro_f1 = compute_macro_precision_f1(
        y_stream, predicted_labels, strict_counted_flags, EXPECTED_NUM_CLASSES
    )
    enrollment_aware_macro_precision, enrollment_aware_macro_f1 = compute_macro_precision_f1(
        y_stream, predicted_labels, enrollment_aware_counted_flags, EXPECTED_NUM_CLASSES
    )

    prediction_log = pd.DataFrame({
        "sample_index": np.arange(num_samples, dtype=np.int32),
        "true_label": y_stream.astype(np.int32, copy=False),
        "predicted_label": predicted_labels,
        "correct": correct_flags,
        "seen_class_before": seen_class_before_flags,
        "counted_in_strict_metric": strict_counted_flags,
        "counted_in_enrollment_aware_metric": enrollment_aware_counted_flags,
        "true_score": true_scores,
        "max_non_true_score": max_non_true_scores,
        "running_strict_accuracy": running_strict_accuracy,
        "running_enrollment_aware_accuracy": running_enrollment_aware_accuracy,
        "ridge_lambda_used": ridge_lambda,
        "activation": ACTIVATION,
    })
    prediction_log.to_csv(prediction_log_path, index=False)

    np.savez_compressed(
        score_output_path,
        true_labels=y_stream.astype(np.int32, copy=False),
        predicted_labels=predicted_labels,
        correct_flags=correct_flags,
        seen_class_before_flags=seen_class_before_flags,
        true_scores=true_scores,
        max_non_true_scores=max_non_true_scores,
        strict_counted_flags=strict_counted_flags,
        enrollment_aware_counted_flags=enrollment_aware_counted_flags,
    )

    contains_nan_scores = bool(np.isnan(true_scores).any() or np.isnan(max_non_true_scores).any())
    contains_inf_scores = bool(np.isinf(true_scores).any() or np.isinf(max_non_true_scores).any())
    runtime_seconds = float(time.time() - experiment_start)

    notes = "V6 Step 7 RP-only RanPAC run; no no-RP baseline used; fixed V6 RP output consumed."
    if contains_nan_scores:
        notes += " NaN scores are expected before any class has been seen."
    if fallback_note:
        notes += " " + fallback_note

    return {
        "experiment_id": experiment_id,
        "feature_setting_full": str(row["feature_setting_full"]),
        "feature_alias": str(row["feature_alias"]),
        "rp_method_full": str(row["rp_method_full"]),
        "rp_alias": str(row["rp_alias"]),
        "projection_dim": projection_dim,
        "output_dim": output_dim,
        "seed": seed,
        "seed_label": seed_label,
        "is_rp": True,
        "is_baseline": False,
        "strict_prequential_accuracy": strict_prequential_accuracy,
        "enrollment_aware_accuracy": enrollment_aware_accuracy,
        "strict_macro_precision": strict_macro_precision,
        "strict_macro_f1": strict_macro_f1,
        "enrollment_aware_macro_precision": enrollment_aware_macro_precision,
        "enrollment_aware_macro_f1": enrollment_aware_macro_f1,
        "runtime_seconds": runtime_seconds,
        "ridge_lambda_initial": RIDGE_LAMBDA_INITIAL,
        "ridge_lambda_used": ridge_lambda,
        "ridge_fallback_used": bool(ridge_fallback_used),
        "activation": ACTIVATION,
        "prediction_log_path": str(prediction_log_path),
        "score_output_path": str(score_output_path),
        "contains_nan_scores": contains_nan_scores,
        "contains_inf_scores": contains_inf_scores,
        "notes": notes,
        "fresh_ranpac_state": True,
        "internal_rp_generated": False,
        "source_rp_output_path": str(x_rp_path),
        "num_seen_classes_final": int(num_seen_classes),
    }


def run_ranpac_with_fallback(row, y_stream):
    try:
        return run_ranpac_once(row, y_stream, RIDGE_LAMBDA_INITIAL, ridge_fallback_used=False)
    except FloatingPointError as first_error:
        fallback_note = (
            f"Initial ridge_lambda={RIDGE_LAMBDA_INITIAL} failed with: {first_error}. "
            f"Re-ran with ridge_lambda={RIDGE_LAMBDA_FALLBACK}."
        )
        print("Ridge fallback triggered:", row["experiment_id"])
        print(fallback_note)
        return run_ranpac_once(
            row,
            y_stream,
            RIDGE_LAMBDA_FALLBACK,
            ridge_fallback_used=True,
            fallback_note=fallback_note,
        )


## 4. Run 135 independent RP-only RanPAC experiments

Each row is evaluated independently with a fresh RanPAC state. The output names are concise and based on the V6 experiment_id.


In [14]:
all_results = []
step7_start = time.time()

for idx, row in rp_summary.iterrows():
    label = f"[{idx + 1:03d}/{len(rp_summary):03d}] {row['experiment_id']} | {row['feature_setting_full']} | {row['rp_method_full']} | dim={row['projection_dim']}"
    print("\n" + label)
    print("source:", row["output_path"])
    result = run_ranpac_with_fallback(row, y_stream)
    all_results.append(result)
    print(
        "strict_accuracy =", round(result["strict_prequential_accuracy"], 6),
        "enrollment_aware_accuracy =", round(result["enrollment_aware_accuracy"], 6),
        "ridge_lambda_used =", result["ridge_lambda_used"],
        "runtime_seconds =", round(result["runtime_seconds"], 2),
    )

summary_df = pd.DataFrame(all_results)
summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False)

step7_runtime_seconds = time.time() - step7_start
print("\nV6 Step 7 completed.")
print("Total runtime seconds:", round(step7_runtime_seconds, 2))
print("Summary saved to:", SUMMARY_OUTPUT_PATH)



[001/135] both_bg_d100_s27 | both_pqrst_resnet1d | bernoulli_gaussian | dim=100
source: F:\ECG\data\processed\rp_v6\seed27\X_both_bg_d100_s27.npy
strict_accuracy = 0.508379 enrollment_aware_accuracy = 0.523144 ridge_lambda_used = 0.001 runtime_seconds = 3.94

[002/135] both_bg_d200_s27 | both_pqrst_resnet1d | bernoulli_gaussian | dim=200
source: F:\ECG\data\processed\rp_v6\seed27\X_both_bg_d200_s27.npy
strict_accuracy = 0.621444 enrollment_aware_accuracy = 0.639494 ridge_lambda_used = 0.001 runtime_seconds = 6.7

[003/135] both_bg_d500_s27 | both_pqrst_resnet1d | bernoulli_gaussian | dim=500
source: F:\ECG\data\processed\rp_v6\seed27\X_both_bg_d500_s27.npy
strict_accuracy = 0.750852 enrollment_aware_accuracy = 0.77266 ridge_lambda_used = 0.001 runtime_seconds = 38.36

[004/135] both_gauss_d100_s27 | both_pqrst_resnet1d | gaussian | dim=100
source: F:\ECG\data\processed\rp_v6\seed27\X_both_gauss_d100_s27.npy
strict_accuracy = 0.508268 enrollment_aware_accuracy = 0.52303 ridge_lambda_us

## 5. Final validation

The final validation follows AGENT.md section 26.9 exactly and confirms that no no-RP experiment was run.


In [17]:
summary_df = pd.read_csv(SUMMARY_OUTPUT_PATH)

required_score_arrays = [
    "true_labels",
    "predicted_labels",
    "correct_flags",
    "seen_class_before_flags",
    "true_scores",
    "max_non_true_scores",
    "strict_counted_flags",
    "enrollment_aware_counted_flags",
]

prediction_logs_exist = bool(summary_df["prediction_log_path"].map(lambda path: Path(str(path)).exists()).all())
score_outputs_exist = bool(summary_df["score_output_path"].map(lambda path: Path(str(path)).exists()).all())

required_score_arrays_exist = True
for score_path in summary_df["score_output_path"].astype(str):
    if not Path(score_path).exists():
        required_score_arrays_exist = False
        break
    score_data = np.load(score_path, allow_pickle=False)
    if not all(array_name in score_data.files for array_name in required_score_arrays):
        required_score_arrays_exist = False
        break

metric_columns = [
    "strict_prequential_accuracy",
    "enrollment_aware_accuracy",
    "strict_macro_f1",
    "enrollment_aware_macro_f1",
]
all_metrics_finite = bool(np.isfinite(summary_df[metric_columns].to_numpy(dtype=float)).all())

no_norp_experiments = bool(
    (len(summary_df) == EXPECTED_RP_EXPERIMENT_ROWS)
    and summary_df["is_rp"].astype(bool).all()
    and (~summary_df["is_baseline"].astype(bool)).all()
    and (summary_df["rp_alias"].astype(str) != "norp").all()
    and (summary_df["seed_label"].astype(str) != "norp").all()
)

ridge_fallback_count = int(summary_df["ridge_fallback_used"].astype(bool).sum())

validation = {
    "original_step6_summary_rows": original_step6_rows,
    "rp_only_rows_selected": rp_only_rows_selected,
    "no_rp_rows_skipped": no_rp_rows_skipped,
    "expected_rp_experiment_rows": EXPECTED_RP_EXPERIMENT_ROWS,
    "actual_experiment_rows": int(len(summary_df)),
    "prediction_logs_exist": prediction_logs_exist,
    "score_outputs_exist": score_outputs_exist,
    "required_score_arrays_exist": required_score_arrays_exist,
    "summary_csv_exists": SUMMARY_OUTPUT_PATH.exists(),
    "all_strict_enrollment_metrics_finite": all_metrics_finite,
    "no_no_rp_experiments_were_run": no_norp_experiments,
    "ridge_fallback_count": ridge_fallback_count,
    "summary_output_path": str(SUMMARY_OUTPUT_PATH),
    "output_dir": str(OUTPUT_DIR),
}

with open(VALIDATION_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2, ensure_ascii=False)

final_validation_text = "\n".join([
    f"Original Step 6 summary rows = {original_step6_rows}",
    f"RP-only rows selected = {rp_only_rows_selected}",
    f"No-RP rows skipped = {no_rp_rows_skipped}",
    f"Expected RP experiment rows = {EXPECTED_RP_EXPERIMENT_ROWS}",
    f"Actual experiment rows = {len(summary_df)}",
    f"Prediction logs exist = {prediction_logs_exist}",
    f"Score outputs exist = {score_outputs_exist}",
    f"Required score arrays exist = {required_score_arrays_exist}",
    f"Summary CSV exists = {SUMMARY_OUTPUT_PATH.exists()}",
    f"All strict/enrollment metrics finite = {all_metrics_finite}",
    f"No no-RP experiments were run = {no_norp_experiments}",
])

print(final_validation_text)
print("Ridge fallback count =", ridge_fallback_count)
print("Validation report path =", VALIDATION_REPORT_PATH)


Original Step 6 summary rows = 138
RP-only rows selected = 135
No-RP rows skipped = 3
Expected RP experiment rows = 135
Actual experiment rows = 135
Prediction logs exist = True
Score outputs exist = True
Required score arrays exist = True
Summary CSV exists = True
All strict/enrollment metrics finite = True
No no-RP experiments were run = True
Ridge fallback count = 0
Validation report path = F:\ECG\data\processed\ranpac_v6\ranpac_v6_validation_report.json
